In [1]:
import jax
import jax.numpy as jnp
from flax import linen as nn
from flax.training import train_state
import optax
from tqdm import tqdm
from pathlib import Path
import urllib.request

In [2]:
data_path = Path("data")
data_path.mkdir(exist_ok=True)
file_path = data_path / "input.txt"
if not file_path.exists():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, file_path)

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

In [3]:
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

In [4]:
data = jnp.array(encode(text), dtype=jnp.int32)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


In [5]:
block_size = 256
batch_size = 64
emb_dim = 384
num_heads = 12
num_layers = 12
dropout_rate = 0.1
max_lr = 3e-4
weight_decay = 0.01
total_steps = 10000
eval_iters = 100

In [6]:
class GPT(nn.Module):
    vocab_size: int
    emb_dim: int
    block_size: int
    num_heads: int
    num_layers: int
    dropout_rate: float

    def setup(self):
        # Token + positional embeddings
        self.token_embed = nn.Embed(self.vocab_size, self.emb_dim)
        self.pos_embed = self.param(
            "pos_embed", nn.initializers.normal(0.02), (1, self.block_size, self.emb_dim)
        )
        # Dropout for embeddings (optional, common practice)
        self.dropout_emb = nn.Dropout(rate=self.dropout_rate)

        # Decoder layers
        self.layers = [
            DecoderBlock(self.emb_dim, self.num_heads, self.dropout_rate)
            for _ in range(self.num_layers)
        ]
        self.ln_f = nn.LayerNorm()
        self.lm_head = nn.Dense(self.vocab_size, use_bias=False)

    def __call__(self, idx, deterministic=True):
        B, T = idx.shape
        # Token embeddings + positional embeddings
        tok_emb = self.token_embed(idx)                     # (B, T, emb_dim)
        pos_emb = self.pos_embed[:, :T, :]                  # (1, T, emb_dim)
        x = tok_emb + pos_emb
        x = self.dropout_emb(x, deterministic=deterministic)

        # Pass through decoder blocks
        for layer in self.layers:
            x = layer(x, deterministic=deterministic)

        x = self.ln_f(x)
        logits = self.lm_head(x)                            # (B, T, vocab_size)
        return logits

In [ ]:

class DecoderBlock(nn.Module):
    emb_dim: int
    num_heads: int
    dropout_rate: float

    def setup(self):
        self.ln1 = nn.LayerNorm()
        self.attn = nn.MultiHeadDotProductAttention(
            num_heads=self.num_heads,
            qkv_features=self.emb_dim,
            dropout_rate=self.dropout_rate,
            deterministic=False,   # will be overridden in __call__
        )
        self.dropout_attn = nn.Dropout(rate=self.dropout_rate)
        self.ln2 = nn.LayerNorm()
        self.mlp = MLP(self.emb_dim, self.dropout_rate)

    def __call__(self, x, deterministic=True):
        # Pre-LN architecture
        residual = x
        x = self.ln1(x)
        # Causal attention (mask future positions)
        x = self.attn(x, x, x, mask=self._causal_mask(x.shape[1]), deterministic=deterministic)
        x = self.dropout_attn(x, deterministic=deterministic)
        x = residual + x

        residual = x
        x = self.ln2(x)
        x = self.mlp(x, deterministic=deterministic)
        x = residual + x
        return x

    def _causal_mask(self, T):
        mask = jnp.tril(jnp.ones((T, T)))
        return mask[jnp.newaxis, jnp.newaxis, :, :]  # (1, 1, T, T)


In [ ]:
class MLP(nn.Module):
    emb_dim: int
    dropout_rate: float

    def setup(self):
        self.fc1 = nn.Dense(4 * self.emb_dim)
        self.gelu = nn.gelu
        self.dropout = nn.Dropout(rate=self.dropout_rate)
        self.fc2 = nn.Dense(self.emb_dim)

    def __call__(self, x, deterministic=True):
        x = self.fc1(x)
        x = self.gelu(x)
        x = self.dropout(x, deterministic=deterministic)
        x = self.fc2(x)
        return x

In [ ]:
def get_batch(rng, split, train_data, val_data, batch_size, block_size):
    data = train_data if split == "train" else val_data
    n = len(data) - block_size
    indices = jax.random.randint(rng, (batch_size,), 0, n)
    x = jnp.stack([data[i:i+block_size] for i in indices])
    y = jnp.stack([data[i+1:i+block_size+1] for i in indices])
    return x, y

In [ ]:
def cross_entropy_loss(logits, targets):
    # logits: (B, T, V), targets: (B, T)
    B, T, V = logits.shape
    logits = logits.reshape(-1, V)
    targets = targets.reshape(-1)
    return optax.softmax_cross_entropy_with_integer_labels(logits, targets).mean()

def compute_loss(params, rng, batch, deterministic):
    logits = model.apply(params, batch[0], deterministic=deterministic, rngs={"dropout": rng})
    loss = cross_entropy_loss(logits, batch[1])
    return loss, logits

In [ ]:
@jax.jit
def train_step(state, rng, batch):
    def loss_fn(params):
        loss, _ = compute_loss(params, rng, batch, deterministic=False)
        return loss
    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss